In [244]:
import cv2
import numpy as np
from skimage.morphology import binary_opening, binary_closing, remove_small_objects, remove_small_holes, disk
from skimage.measure import label, regionprops
import pandas as pd

import pathlib
from PIL import Image

In [245]:
def guardar(img_ary, carpeta, nombre_archivo):
    carpeta = pathlib.Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)

    ruta = carpeta / nombre_archivo
    Image.fromarray(img_ary).save(ruta)

    return ruta

In [246]:
def convolucion(img, kernel):

    inH, inW = img.shape
    kH, kW = kernel.shape

    pad_h = kH // 2
    pad_w = kW // 2

    img_pad = np.pad(img.astype(np.float64), ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    salida = np.zeros((inH, inW))
    kernel_flip = np.flipud(np.fliplr(kernel))

    for i in range(inH):
        for j in range(inW):

            region = img_pad[i:i+kH, j:j+kW]
            salida[i, j] = np.sum(region * kernel_flip)

    return salida

In [247]:
def ecualizar(img):
    hist, _ = np.histogram(img.flatten(), 256, [0, 256])
    histSum = np.cumsum(hist).astype(np.float64)
    histSum /= histSum[-1]

    img_eq = (histSum[img] * 255).astype(np.uint8)

    return img_eq

In [248]:
def trans_log(img_eq):
    img_eq = img_eq.astype(np.float32)

    c = 255 / np.log1p(np.max(img_eq))
    img_log = c * np.log1p(img_eq)

    img_log = np.array(img_log, dtype=np.uint8)

    return img_log

In [249]:
def trans_gamma(img_eq):
    img_eq_g = img_eq.astype(np.float32) / 255.0

    gamma = 0.7
    img_gamma = (np.power(img_eq_g, gamma) * 255).astype(np.uint8)

    return img_gamma

In [250]:
def gauss_blur(img_log, img_gamma):
    img_suave_log = cv2.GaussianBlur(img_log, (9, 9), 1.5)
    img_suave_gamma = cv2.GaussianBlur(img_gamma, (5, 5), 1)

    return img_suave_log, img_suave_gamma

In [251]:
def binarizacion(img_suave_gamma, img_eq):

    umbral = np.percentile(img_suave_gamma, 40)
    mask = img_suave_gamma < umbral

    mask = binary_opening(mask, disk(3))
    mask = remove_small_objects(mask, min_size=500)
    mask = binary_closing(mask, disk(3))
    mask_clean = remove_small_holes(mask, area_threshold=2000)

    img_eq = img_eq.astype(np.uint8)
    img_rgb = cv2.cvtColor(img_eq, cv2.COLOR_GRAY2RGB)

    overlayBin = img_rgb.copy()

    overlayBin[~mask_clean] = [255, 0, 0]
    return mask_clean, overlayBin

In [252]:
def roi(mask, img_eq):

    labels = label(mask)
    conteo = np.bincount(labels.ravel())
    conteo[0] = 0
    label_ganador = np.argmax(conteo)
    mask_final = labels == label_ganador

    img = cv2.cvtColor(img_eq, cv2.COLOR_GRAY2BGR)
    overlay = img.copy()
    overlay[~mask_final] = [255, 0, 0]

    return mask_final, overlay

In [253]:
knlap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)


def laplaciano(img):
    lap = convolucion(img, knlap)
    lap_abs = np.abs(lap)

    p_low = np.percentile(lap_abs, 0.5)
    p_high = np.percentile(lap_abs, 99.5)
    lap_clip = np.clip(lap_abs, p_low, p_high)
    lap_norm = ((lap_clip - p_low) * 255.0 / (p_high - p_low)).astype(np.uint8)

    return lap_norm

In [254]:
kn = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float64)
kw = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float64)


def prewitt(img):
    resultado_x = convolucion(img, kn)
    resultado_y = convolucion(img, kw)
    mag = np.sqrt(resultado_x**2 + resultado_y**2)

    p_low = np.percentile(mag, 0.5)
    p_high = np.percentile(mag, 99.5)
    mag_clip = np.clip(mag, p_low, p_high)
    mag_norm = ((mag_clip - p_low) * 255.0 / (p_high - p_low)).astype(np.uint8)

    return mag_norm

In [260]:
# INGESTA
dataset = pathlib.Path("C:/Users/Sebastian/Dropbox/My PC (LAPTOP-1ULHO96H)/Desktop/archive/data/validation/nml")
res = pathlib.Path("C:/Users/Sebastian/Dropbox/My PC (LAPTOP-1ULHO96H)/Desktop/res")
res.mkdir(exist_ok=True)

rutas = sorted(dataset.glob("*.png"))
print(f"Imágenes encontradas: {len(rutas)}")

registros = []
for ruta in rutas:
    img_pil = Image.open(ruta)
    w, h = img_pil.size
    registros.append({
        'id': ruta.stem,
        'ruta': str(ruta),
        'carpeta': ruta.parent.name,
        'formato': ruta.suffix.lower(),
        'ancho': w,
        'alto': h,
        'tamaño_kb': round(ruta.stat().st_size / 1024, 2)
    })

manifest = pd.DataFrame(registros)
manifest.to_csv(res / 'manifest.csv', index=False)
print("\n manifest.csv guardado:")
manifest.head()

Imágenes encontradas: 130

 manifest.csv guardado:


,id,ruta,carpeta,formato,ancho,alto,tamaño_kb
0,1003_im00038_nml_0,C:\Users\Sebastian\Dropbox\My PC (LAPTOP-1ULHO...,nml,.png,129,299,47.23
1,1008_im01170_nml_0,C:\Users\Sebastian\Dropbox\My PC (LAPTOP-1ULHO...,nml,.png,402,198,77.41
2,1020_im01238_nml_0,C:\Users\Sebastian\Dropbox\My PC (LAPTOP-1ULHO...,nml,.png,182,201,44.01
3,1050_im00315_nml_0,C:\Users\Sebastian\Dropbox\My PC (LAPTOP-1ULHO...,nml,.png,224,314,90.34
4,1053_im01003_nml_0,C:\Users\Sebastian\Dropbox\My PC (LAPTOP-1ULHO...,nml,.png,211,199,42.45


In [256]:
imagenes_gray = {}

for ruta in rutas:
    img_id = ruta.stem
    carpeta_img = res / img_id
    carpeta_img.mkdir(parents=True, exist_ok=True)

    img_gray = cv2.imread(str(ruta), cv2.IMREAD_GRAYSCALE)

    imagenes_gray[img_id] = img_gray

In [257]:
imgs_eq = {}
imgs_log = {}
imgs_gamma = {}
imgs_suave_log = {}
imgs_suave_gamma = {}
imgs_mask_raw = {}
imgs_mask_clean = {}
imgs_overlay = {}
imgs_mask_final = {}
imgs_prew = {}
imgs_lap = {}

for ruta in rutas:
    img_id = ruta.stem
    carpeta = res / img_id
    carpeta.mkdir(exist_ok=True)

    img = imagenes_gray[img_id]

    # Step 1
    guardar(img, carpeta, "step_01_input.png")

    # Step 2
    img_eq = ecualizar(img)
    imgs_eq[img_id] = img_eq
    guardar(img_eq, carpeta, "step_02_homogenized.png")

    # Step 3
    img_log = trans_log(img_eq)
    img_gamma = trans_gamma(img_eq)
    imgs_log[img_id] = img_log
    imgs_gamma[img_id] = img_gamma
    guardar(img_log, carpeta, "step_03_transform1_log.png")
    guardar(img_gamma, carpeta, "step_03_transform2_gamma.png")

    # Suavizado
    img_suave_log, img_suave_gamma = gauss_blur(img_log, img_gamma)
    imgs_suave_log[img_id] = img_suave_log
    imgs_suave_gamma[img_id] = img_suave_gamma

    # Step 4
    mask_clean, overlayBin = binarizacion(img_suave_gamma, img_eq)
    imgs_mask_raw[img_id] = img_suave_gamma < np.percentile(img_suave_gamma, 40)
    imgs_mask_clean[img_id] = mask_clean
    imgs_overlay[img_id] = overlayBin
    guardar((imgs_mask_raw[img_id] * 255).astype(np.uint8), carpeta, "step_04_binary_raw.png")
    guardar((mask_clean * 255).astype(np.uint8), carpeta, "step_04_binary_clean.png")
    guardar(overlayBin, carpeta, "step_04_overlay.png")

    # Step 5
    mask_final, overlay_roi = roi(mask_clean, img_eq)
    imgs_mask_final[img_id] = mask_final
    guardar((mask_final * 255).astype(np.uint8), carpeta, "step_05_roi_selected.png")
    guardar(overlay_roi, carpeta, "step_05_roi_overlay.png")

    # Step 6
    img_para_bordes = cv2.GaussianBlur(img_log, (3, 3), sigmaX=0.8)
    prew = prewitt(img_para_bordes)
    prew_norm = (prew / prew.max() * 255).astype(np.uint8)
    imgs_prew[img_id] = prew_norm
    guardar(prew_norm, carpeta, "step_06_grad_mag.png")

    # Step 7
    lap = laplaciano(img_para_bordes)
    imgs_lap[img_id] = lap
    guardar(lap, carpeta, "step_07_laplacian.png")

print(f"Pipeline completo {len(rutas)} imágenes procesadas")

Pipeline completo 130 imágenes procesadas
